# ORCA — Train QLoRA tài chính đa mảng (Colab T4)

**Cắm vào là train:** notebook tự sinh dataset đa mảng rồi QLoRA Qwen2.5-7B.

| Mảng | Có |
|------|-----|
| Thị trường VN | yes |
| Cổ phiếu + so sánh | yes |
| Ngành | yes |
| Hàng hóa | yes |
| Crypto | yes |
| Từ chối advice mua/bán | yes |

**Runtime → Change runtime type → T4 GPU** rồi chạy hết cell.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Bật Runtime → T4 GPU"
print(torch.cuda.get_device_name(0))


In [ ]:
!pip install -q -U transformers==4.51.3 peft==0.15.2 trl==0.15.2 bitsandbytes==0.45.4 accelerate datasets sentencepiece protobuf


## 1. Sinh dataset đa mảng sẵn


In [ ]:
import urllib.request, json
from pathlib import Path
from collections import Counter

GEN_URL = "https://raw.githubusercontent.com/duyanhphan13579dz-dot/Orca-Multi-Finance/main/datasets/llm-human-voice/scripts/generate_multidomain_dataset.py"
urllib.request.urlretrieve(GEN_URL, "generate_multidomain_dataset.py")
!python generate_multidomain_dataset.py sft_multidomain_finance.jsonl

rows = [json.loads(l) for l in open("sft_multidomain_finance.jsonl", encoding="utf-8") if l.strip()]
print("N samples:", len(rows))
print("Branches:", dict(Counter(r["branch"] for r in rows)))
print("Depths:", dict(Counter(r["depth"] for r in rows)))


## 2. Đổi sang chat format (system đa mảng)


In [ ]:
SYSTEM = """Bạn là ORCA Agent — chuyên gia phân tích tài chính đa thị trường
(chứng khoán Việt Nam, ngành, hàng hóa, crypto, forex, lãi suất, vĩ mô).
Chỉ dùng số liệu trong CONTEXT/NARRATIVE và TÀI LIỆU TRUY XUẤT.
Không bịa số, không khuyến nghị mua/bán tuyệt đối, không target giá khi không có trong hệ thống.
Giọng senior research analyst: rõ ràng, có chuyển tiếp, mỗi số kèm ý nghĩa."""

def format_retrieved(retrieved):
    if not retrieved:
        return "(Không có đoạn tài liệu bổ sung.)"
    parts = []
    for i, p in enumerate(retrieved, 1):
        src = p.get("source", "?")
        title = p.get("title", "")
        content = p.get("content", "")
        parts.append(f"[{i}] source={src} {title}\n{content}")
    return "\n\n".join(parts)

chat = []
for rec in rows:
    narrative = (rec.get("context_narrative") or "")[:8000]
    contract = json.dumps(rec.get("context_contract") or {}, ensure_ascii=False)[:8000]
    user = (
        f"CÂU HỎI: {rec['question']}\n\n"
        f"NARRATIVE:\n{narrative}\n\n"
        f"CONTEXT JSON:\n{contract}\n\n"
        f"TÀI LIỆU TRUY XUẤT:\n{format_retrieved(rec.get('retrieved') or [])}\n\n"
        "Viết phân tích tài chính chuyên sâu đa mảng, bám số liệu."
    )
    chat.append({
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user},
            {"role": "assistant", "content": rec.get("preferred_answer") or ""},
        ]
    })

Path("sft_multidomain_chat.jsonl").write_text(
    "\n".join(json.dumps(c, ensure_ascii=False) for c in chat) + "\n",
    encoding="utf-8",
)
print("chat examples", len(chat))


## 3. Load Qwen2.5-7B 4-bit + LoRA r=32 (tối ưu finance)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen2.5-7B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ),
)
model.print_trainable_parameters()


## 4. Train 3 epoch


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

ds = load_dataset("json", data_files="sft_multidomain_chat.jsonl", split="train")

def formatting_func(ex):
    return tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )

args = SFTConfig(
    output_dir="./orca-multidomain-out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    max_seq_length=2048,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)
trainer.train()
print("TRAIN DONE")


## 5. Lưu + download adapter


In [ ]:
OUT = "orca-analyst-lora"
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
!zip -r orca-analyst-lora.zip orca-analyst-lora
from google.colab import files
files.download("orca-analyst-lora.zip")


## 6. Deploy vLLM + ORCA

```bash
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --enable-lora \
  --lora-modules orca-analyst-v1=./orca-analyst-lora \
  --host 0.0.0.0 --port 8000 --max-model-len 4096
```

```bash
AI_BASE_URL=https://YOUR_HOST/v1
AI_MODEL_ANALYSIS=orca-analyst-v1
AI_MODEL_ANALYSIS_FALLBACKS=qwen/qwen3.8-27b:free,inclusionai/ling-3.0-flash-fin:free
```

Xem thêm: `deploy/README.md`, `configs/lora_finance.yaml`
